In [ ]:
# Saving figures for the book (website and print).
# save_for_print(fig, name, width_in) makes sure that no text is smaller than
# MIN_PT points once the figure is printed width_in inches wide (the book's text
# is 4.5 in wide, so a figure at 90% is 4.05 in), hides the tick numbers of 3D
# axes, and saves img/<name>.png at PRINT_DPI for that printed width.
import os
import matplotlib.text
from mpl_toolkits.mplot3d import Axes3D

IMG_DIR = 'img'      # where the images go (the book's img/ folder)
MIN_PT = 7           # smallest text size in print, in points
PRINT_DPI = 400      # resolution at the printed size

def save_for_print(fig, name, width_in, min_pt=MIN_PT, hide_3d_ticks=True):
    if hide_3d_ticks:
        for ax in fig.axes:
            if isinstance(ax, Axes3D):
                ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
    for _ in range(3):  # enlarging text changes the figure's extent, so repeat
        fig.canvas.draw()
        scale = width_in / fig.get_tightbbox().width   # printed size / drawn size
        texts = [t for t in fig.findobj(matplotlib.text.Text)
                 if t.get_visible() and t.get_text().strip()]
        small = [t for t in texts if t.get_fontsize() * scale < min_pt - 0.05]
        for t in small:
            t.set_fontsize(min_pt / scale)
        for ax in fig.axes:  # tick labels are re-created when drawn, so set them on the axes too
            for axis in ('x', 'y'):
                labels = getattr(ax, f'get_{axis}ticklabels')()
                if labels and labels[0].get_fontsize() * scale < min_pt - 0.05:
                    ax.tick_params(axis=axis, labelsize=min_pt / scale)
        if not small:
            break
    fig.canvas.draw()
    dpi = PRINT_DPI * width_in / fig.get_tightbbox().width
    os.makedirs(IMG_DIR, exist_ok=True)
    fig.savefig(os.path.join(IMG_DIR, name + '.png'), dpi=dpi, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    print(f'{name}.png: {width_in} in wide in print, smallest text {min_pt} pt')


#Particle in a Ring

## Eigenfunctions and Eigenvalues

Now, let's explore the quantum mechanics of a particle confined to a ring. We will consider a ring of radius $R$ where the particle's position is described by an angle $\phi$ from $0$ to $2\pi$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from matplotlib.colors import Normalize

# --- Define physical parameters for the ring ---
R = 1.0  # Radius of the ring (e.g., 1 meter)
# Mass (m) and h_bar are already defined from the particle in a box section

# --- Function to calculate eigenfunctions for a particle in a ring ---
def particle_in_ring_eigenfunction(phi, m_quantum, R):
    """Calculates the m_quantum-th eigenfunction for a particle in a ring.

    Args:
        phi (numpy.ndarray): Array of angular points.
        m_quantum (int): Quantum number (m = 0, ±1, ±2, ...).
        R (float): Radius of the ring.

    Returns:
        numpy.ndarray: The complex value of the eigenfunction at each phi.
    """
    # Note: For normalization, we use 1/sqrt(2*pi*R) if we consider the arc length,
    # or 1/sqrt(2*pi) if we normalize over angle phi.
    # Here we normalize over phi for simplicity, considering the domain [0, 2pi].
    return (1 / np.sqrt(2 * np.pi)) * np.exp(1j * m_quantum * phi)

# --- Function to calculate eigenvalues for a particle in a ring ---
def particle_in_ring_eigenvalue(m_quantum, R, m_mass, h_bar):
    """Calculates the m_quantum-th eigenvalue (energy) for a particle in a ring.

    Args:
        m_quantum (int): Quantum number (m = 0, ±1, ±2, ...).
        R (float): Radius of the ring.
        m_mass (float): Mass of the particle.
        h_bar (float): Reduced Planck's constant.

    Returns:
        float: The m_quantum-th energy eigenvalue.
    """
    # I = m_mass * R**2 is the moment of inertia for a point mass on a ring
    return (m_quantum**2 * h_bar**2) / (2 * m_mass * R**2)

# --- Angular grid for visualization ---
phi = np.linspace(0, 2 * np.pi, 500) # 500 points from 0 to 2*pi

# --- Quantum numbers to visualize for the ring ---
quantum_numbers_ring = [0, 1, -1, 2, -2]


## Combined 3D Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from matplotlib.colors import Normalize

plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif', 'serif']
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'cm' # Use Computer Modern for math text, which is serif

# Ensure variables from previous cells are available
# R, phi, quantum_numbers_ring, particle_in_ring_eigenfunction

# Convert polar coordinates to Cartesian for the ring
x_coords = R * np.cos(phi)
y_coords = R * np.sin(phi)

# Calculate the maximum possible amplitude of the normalized wavefunction
max_possible_wavefunction_amplitude = 1 / np.sqrt(2 * np.pi)

# Define a global normalization for the colormap to ensure consistent coloring
global_norm = Normalize(vmin=-max_possible_wavefunction_amplitude, vmax=max_possible_wavefunction_amplitude)
cmap = cm.RdBu_r # Red for positive, Blue for negative, white/light in middle

fig = plt.figure(figsize=(15, 6 * len(quantum_numbers_ring)), constrained_layout=True) # Increased width to 50 for max space, added constrained_layout
#fig.suptitle('Combined 3D Visualization of Particle in a Ring Eigenfunctions', fontsize=16)

wavefunction_linewidth = 4

# Determine grid size for subplots
num_quantum_numbers = len(quantum_numbers_ring)
rows = num_quantum_numbers # Each m_q will have one row of (Real, Imaginary)
cols = 2

# Define desired tick locations for the 3D plots
tick_locations = np.arange(-1.0, 1.1, 0.5)

for i, m_q in enumerate(quantum_numbers_ring):
    # Calculate the eigenfunction (complex result)
    psi_m = particle_in_ring_eigenfunction(phi, m_q, R)

    # --- Real Part Subplot ---
    ax_real = fig.add_subplot(rows, cols, (i * cols) + 1, projection='3d') # First column
    z_coords_real = np.real(psi_m)

    # Plotting with gradient color by plotting small segments
    for k in range(len(phi) - 1):
        ax_real.plot(x_coords[k:k+2], y_coords[k:k+2], z_coords_real[k:k+2],
                color=cmap(global_norm(z_coords_real[k])),
                linewidth=wavefunction_linewidth)

    # Add vertical lines from the ring to the wavefunction
    for j in range(len(phi)):
        ax_real.plot([x_coords[j], x_coords[j]], [y_coords[j], y_coords[j]], [0, z_coords_real[j]], color='gray', linestyle='-', linewidth=0.5, alpha=0.5)

    ax_real.set_title(rf'$m_\ell$ = {m_q} (Real Part)', y=-0.1, fontsize=30)
    ax_real.set_xlabel(rf'$x$', fontsize=16)
    ax_real.set_ylabel(rf'$y$', fontsize=16)
    #ax_real.set_zlabel(r'$\mathfrak{Re}[\psi(\varphi)]$', fontsize=18, labelpad=35) # Increased labelpad and fontsize
    ax_real.tick_params(axis='x', labelsize=12)
    ax_real.tick_params(axis='y', labelsize=12)
    ax_real.tick_params(axis='z', labelsize=12)
    ax_real.set_xticks(tick_locations)
    ax_real.set_yticks(tick_locations)
    ax_real.set_zticks(tick_locations)
    ax_real.grid(True)
    ax_real.set_zlim(-1, 1)
    ax_real.plot(x_coords, y_coords, np.zeros_like(z_coords_real), color='black', linestyle='-', linewidth=2) # Reference ring

    # --- Imaginary Part Subplot ---
    ax_imag = fig.add_subplot(rows, cols, (i * cols) + 2, projection='3d') # Second column
    z_coords_imag = np.imag(psi_m)

    # Plotting with gradient color by plotting small segments
    for k in range(len(phi) - 1):
        ax_imag.plot(x_coords[k:k+2], y_coords[k:k+2], z_coords_imag[k:k+2],
                color=cmap(global_norm(z_coords_imag[k])),
                linewidth=wavefunction_linewidth)

    # Add vertical lines from the ring to the wavefunction
    for j in range(len(phi)):
        ax_imag.plot([x_coords[j], x_coords[j]], [y_coords[j], y_coords[j]], [0, z_coords_imag[j]], color='gray', linestyle='-', linewidth=0.5, alpha=0.5)

    ax_imag.set_title(rf'$m_\ell$ = {m_q} (Imaginary Part)', y=-0.1, fontsize=30) # Increased title font size, using raw f-string
    ax_imag.set_xlabel(rf'$x$', fontsize=16)
    ax_imag.set_ylabel(rf'$y$', fontsize=16)
#    ax_imag.set_zlabel(r'$\mathfrak{Im}[\psi(\varphi)]$', fontsize=18, labelpad=35) # Increased labelpad and fontsize
    ax_imag.tick_params(axis='x', labelsize=12)
    ax_imag.tick_params(axis='y', labelsize=12) # Apply same fix to imaginary plot
    ax_imag.tick_params(axis='z', labelsize=12)
    ax_imag.set_xticks(tick_locations)
    ax_imag.set_yticks(tick_locations)
    ax_imag.set_zticks(tick_locations)
    ax_imag.grid(True)
    ax_imag.set_zlim(-1, 1)
    ax_imag.plot(x_coords, y_coords, np.zeros_like(z_coords_imag), color='black', linestyle='-', linewidth=2) # Reference ring

save_for_print(fig, 'PIR_3d', 2.25)
plt.show()

## Combined Top-Down 2D Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize

# Ensure variables from previous cells are available in the kernel
# R = 1.0
# phi = np.linspace(0, 2 * np.pi, 500)
# quantum_numbers_ring = [0, 1, -1, 2, -2] # Updated quantum_numbers_ring to include -2
# particle_in_ring_eigenfunction(phi, m_quantum, R)

plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif', 'serif']
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'cm' # Use Computer Modern for math text, which is serif


# Convert polar coordinates to Cartesian for the ring
x_coords = R * np.cos(phi)
y_coords = R * np.sin(phi)

# Calculate the maximum possible amplitude of the normalized wavefunction
max_possible_wavefunction_amplitude = 1 / np.sqrt(2 * np.pi)

# Define a global normalization for the colormap to ensure consistent coloring
global_norm = Normalize(vmin=-max_possible_wavefunction_amplitude, vmax=max_possible_wavefunction_amplitude)
cmap = cm.RdBu_r # Red for positive, Blue for negative, white/light in middle

# Determine grid size for subplots
num_quantum_numbers = len(quantum_numbers_ring)
rows = num_quantum_numbers # Each m_q will have one row of (Real, Imaginary)
cols = 2

fig = plt.figure(figsize=(15, 7 * rows)) # Adjust figure height based on number of quantum numbers
#fig.suptitle('Combined Top-Down 2D Visualization of Particle in a Ring Eigenfunctions', fontsize=16)

for i, m_q in enumerate(quantum_numbers_ring):
    # Calculate the eigenfunction (complex result)
    psi_m = particle_in_ring_eigenfunction(phi, m_q, R)

    # --- Real Part Subplot ---
    ax_real = fig.add_subplot(rows, cols, (i * cols) + 1) # First column
    z_coords_real = np.real(psi_m)

    # Plotting with gradient color by plotting small segments
    for k in range(len(phi)):
        ax_real.scatter(x_coords[k], y_coords[k], color=cmap(global_norm(z_coords_real[k])), s=15)

    # Add nodal diameters for Real Part if m_q is not 0
    if m_q != 0:
        num_nodes = 2 * abs(m_q)
        # Nodes for cos(m*phi) are at (n + 0.5) * pi / |m|
        node_angles_real = (np.arange(num_nodes) + 0.5) * np.pi / abs(m_q)
        for j in range(abs(m_q)):
            angle1 = node_angles_real[j]
            angle2 = node_angles_real[j + abs(m_q)]
            ax_real.plot([R * np.cos(angle1), R * np.cos(angle2)],
                         [R * np.sin(angle1), R * np.sin(angle2)],
                         color='purple', linestyle='-', linewidth=2)

    ax_real.set_title(rf'$m_\ell$ = {m_q} (Real Part)', y=-0.22, fontsize=30)
    ax_real.set_xticklabels([])  # no tick numbers: unreadable at print size
    ax_real.set_yticklabels([])
    ax_real.set_aspect('equal', adjustable='box') # Ensure the ring looks circular
    ax_real.grid(True)
    ax_real.set_xlim(-R, R)
    ax_real.set_ylim(-R, R)

    # --- Imaginary Part Subplot ---
    ax_imag = fig.add_subplot(rows, cols, (i * cols) + 2) # Second column
    z_coords_imag = np.imag(psi_m)

    # Plotting with gradient color by plotting small segments
    for k in range(len(phi)):
        ax_imag.scatter(x_coords[k], y_coords[k], color=cmap(global_norm(z_coords_imag[k])), s=15)

    # Add nodal diameters for Imaginary Part if m_q is not 0
    if m_q != 0:
        num_nodes = 2 * abs(m_q)
        # Nodes for sin(m*phi) are at n * pi / |m|
        node_angles_imag = np.arange(num_nodes) * np.pi / abs(m_q)
        for j in range(abs(m_q)):
            angle1 = node_angles_imag[j]
            angle2 = node_angles_imag[j + abs(m_q)]
            ax_imag.plot([R * np.cos(angle1), R * np.cos(angle2)],
                         [R * np.sin(angle1), R * np.sin(angle2)],
                         color='purple', linestyle='-', linewidth=2)



    ax_imag.set_title(rf'$m_\ell$ = {m_q} (Imaginary Part)', y=-0.22, fontsize=30)
    ax_imag.set_xticklabels([])  # no tick numbers: unreadable at print size
    ax_imag.set_yticklabels([])
    ax_imag.set_aspect('equal', adjustable='box') # Ensure the ring looks circular
    ax_imag.grid(True)
    ax_imag.set_xlim(-R, R)
    ax_imag.set_ylim(-R, R)

plt.tight_layout(pad=2.0) # Add padding between subplots
save_for_print(fig, 'PIR_top', 2.25)
plt.show()

## Combined Polar Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize

# Ensure variables from previous cells are available in the kernel
# R = 1.0
# phi = np.linspace(0, 2 * np.pi, 500)
# quantum_numbers_ring = [0, 1, -1, 2]
# particle_in_ring_eigenfunction(phi, m_quantum, R)


plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif', 'serif']
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'cm' # Use Computer Modern for math text, which is serif


# Calculate the maximum possible amplitude of the normalized wavefunction
max_possible_wavefunction_amplitude = 1 / np.sqrt(2 * np.pi)

# Define a global normalization for the colormap to ensure consistent coloring
global_norm = Normalize(vmin=-max_possible_wavefunction_amplitude, vmax=max_possible_wavefunction_amplitude)
cmap = cm.RdBu_r # Red for positive, Blue for negative, white/light in middle

wavefunction_linewidth = 3 # Line width for polar plot
nodal_line_width = 2 # Line width for nodal lines

# Determine grid size for subplots
num_quantum_numbers = len(quantum_numbers_ring)
rows = num_quantum_numbers # Each m_q will have one row of (Real, Imaginary)
cols = 2 # Changed from 3 back to 2 columns

fig_polar_combined = plt.figure(figsize=(15, 7 * rows)) # Adjust figure width for 2 columns
#fig_polar_combined.suptitle('Combined Polar Visualization of Particle in a Ring Eigenfunctions', fontsize=16)

for i, m_q in enumerate(quantum_numbers_ring):
    # Calculate the eigenfunction (complex result)
    psi_m = particle_in_ring_eigenfunction(phi, m_q, R)

    # --- Real Part Polar Subplot ---
    ax_real_polar = fig_polar_combined.add_subplot(rows, cols, (i * cols) + 1, projection='polar') # First column

    # Store the original signed real part for color mapping
    real_part_for_coloring = np.real(psi_m)
    # Get the absolute value of the real part of the wavefunction for plotting as radial coordinate
    radial_coords_real = np.abs(real_part_for_coloring)

    # Plotting with gradient color by plotting small segments
    for k in range(len(phi) - 1):
        ax_real_polar.plot(phi[k:k+2], radial_coords_real[k:k+2],
                      color=cmap(global_norm(real_part_for_coloring[k])),
                      linewidth=wavefunction_linewidth)

    ax_real_polar.set_title(rf'$m_\ell$ = {m_q} (Real Part)', y=-0.15, fontsize=30)
    ax_real_polar.set_xticklabels([]); ax_real_polar.set_yticklabels([])  # no tick labels: unreadable at print size
    ax_real_polar.set_theta_zero_location('E') # Set 0 degrees to the East (right)
    ax_real_polar.set_theta_direction(1)    # Clockwise direction
    r_max = max_possible_wavefunction_amplitude * 1.2 # Extend a bit above max possible
    ax_real_polar.set_rlim(0, r_max) # Radial axis starts from zero

    # Add nodal lines for Real Part if m_q is not 0
    if m_q != 0:
        # Nodes for cos(m*phi) are at (n + 0.5) * pi / |m|
        node_angles_real = []
        for n_val in range(2 * abs(m_q)):
            angle = (n_val + 0.5) * np.pi / abs(m_q)
            if 0 <= angle <= 2 * np.pi:
                node_angles_real.append(angle)
        for node_angle in node_angles_real:
            ax_real_polar.plot([node_angle, node_angle], [0, r_max], color='purple', linestyle='-', linewidth=nodal_line_width)

    # --- Imaginary Part Polar Subplot ---
    ax_imag_polar = fig_polar_combined.add_subplot(rows, cols, (i * cols) + 2, projection='polar') # Second column

    # Store the original signed imaginary part for color mapping
    imaginary_part_for_coloring = np.imag(psi_m)
    # Get the absolute value of the imaginary part of the wavefunction for plotting as radial coordinate
    radial_coords_imag = np.abs(imaginary_part_for_coloring)

    # Plotting with gradient color by plotting small segments
    for k in range(len(phi) - 1):
        ax_imag_polar.plot(phi[k:k+2], radial_coords_imag[k:k+2],
                      color=cmap(global_norm(imaginary_part_for_coloring[k])),
                      linewidth=wavefunction_linewidth)

    ax_imag_polar.set_title(rf'$m_\ell$ = {m_q} (Imaginary Part)', y=-0.15, fontsize=30)
    ax_imag_polar.set_xticklabels([]); ax_imag_polar.set_yticklabels([])  # no tick labels: unreadable at print size
    ax_imag_polar.set_theta_zero_location('E') # Set 0 degrees to the East (right)
    ax_imag_polar.set_theta_direction(1)    # Clockwise direction
    r_max = max_possible_wavefunction_amplitude * 1.2 # Extend a bit above max possible
    ax_imag_polar.set_rlim(0, r_max) # Radial axis starts from zero

    # Add nodal lines for Imaginary Part if m_q is not 0
    if m_q != 0:
        # Nodes for sin(m*phi) are at n * pi / |m|
        node_angles_imag = []
        for n_val in range(2 * abs(m_q)):
            angle = n_val * np.pi / abs(m_q)
            if 0 <= angle <= 2 * np.pi:
                node_angles_imag.append(angle)
        for node_angle in node_angles_imag:
            ax_imag_polar.plot([node_angle, node_angle], [0, r_max], color='purple', linestyle='-', linewidth=nodal_line_width)

plt.tight_layout(rect=[0, 0.03, 1, 0.95], pad=4.0) # Adjust layout to prevent title overlap
save_for_print(fig_polar_combined, 'PIR_polar', 2.25)
plt.show()

# Generate energy level plot


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif', 'serif']
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'cm' # Use Computer Modern for math text, which is serif

plt.rcParams.update({'font.size': 20}) # Global font size increase

# Ensure variables from previous cells are available
# R, m, h_bar, quantum_numbers_ring, particle_in_ring_eigenfunction, particle_in_ring_eigenvalue

# --- Define physical parameters for the ring --- (redefining for a self-contained cell)
R = 1.0  # Radius of the ring (e.g., 1 meter)
m = 1.0  # Mass of the particle (e.g., 1 kg)
h_bar = 1.0  # Reduced Planck's constant (J*s)

# --- Function to calculate eigenfunctions for a particle in a ring --- (redefining)
def particle_in_ring_eigenfunction(phi, m_quantum, R):
    return (1 / np.sqrt(2 * np.pi)) * np.exp(1j * m_quantum * phi)

# --- Function to calculate eigenvalues for a particle in a ring --- (redefining)
def particle_in_ring_eigenvalue(m_quantum, R, m_mass, h_bar):
    return (m_quantum**2 * h_bar**2) / (2 * m_mass * R**2)

# --- Angular grid for visualization --- (redefining)
phi = np.linspace(0, 2 * np.pi, 500) # 500 points from 0 to 2*pi

# --- Quantum numbers to visualize for the ring --- (redefining)
quantum_numbers_ring = [0, 1, -1, 2, -2, 3, -3, 4, -4]

# Collect energies and corresponding m_q values
energy_data = {}
for m_q in quantum_numbers_ring:
    eigenvalue = particle_in_ring_eigenvalue(m_q, R, m, h_bar)
    # Round eigenvalue to handle potential floating point inaccuracies for grouping degeneracies
    rounded_eigenvalue = round(eigenvalue, 5)
    if rounded_eigenvalue not in energy_data:
        energy_data[rounded_eigenvalue] = []
    energy_data[rounded_eigenvalue].append(m_q)

# Sort unique energy values
unique_energies = sorted(energy_data.keys())

# Define the base energy unit for labeling
E_unit = h_bar**2 / (2 * m * R**2)
energy_unit_formula = r'\frac{\hbar^2}{2mr^2}' # The actual formula string to be used in labels

# Create custom y-tick labels
y_tick_labels = []
for energy in unique_energies:
    if energy == 0:
        y_tick_labels.append('0')
    else:
        # Calculate m_quantum^2 corresponding to the energy
        m_squared = int(round(energy / E_unit))
        if m_squared == 1:
            y_tick_labels.append(f'${energy_unit_formula}$')
        else:
            y_tick_labels.append(f'${m_squared} {energy_unit_formula}$')

# --- For polar plot ---
max_possible_wavefunction_amplitude = 1 / np.sqrt(2 * np.pi)
global_norm = Normalize(vmin=-max_possible_wavefunction_amplitude, vmax=max_possible_wavefunction_amplitude)
cmap = cm.RdBu_r
r_max_polar = max_possible_wavefunction_amplitude * 1.5 # Extend a bit above max possible for polar plots


# Create the energy scale plot
fig, ax = plt.subplots(figsize=(24, 18)) # Increased width and height by 1.5 for larger polar plots

# Increased horizontal offset to ensure no overlap, accounting for two polar plots per m_q
offset_step = 2.0 # Adjusted from 1.0, scaled to accommodate larger plots
max_width_needed = 0

for energy in unique_energies:
    m_q_list = sorted(energy_data[energy]) # Sort m_q for consistent side-by-side order
    num_degeneracies = len(m_q_list)

    # Calculate starting x-offset to center the group of lines
    start_offset = -(num_degeneracies - 1) * offset_step / 2

    for idx, m_q in enumerate(m_q_list):
        x_pos = start_offset + idx * offset_step
        ax.hlines(energy, x_pos - 0.25, x_pos + 0.25, colors='blue', linestyles='-', linewidth=4, zorder=2)
        ax.annotate(rf'$m_\ell$ = {m_q}', (x_pos, energy), textcoords="offset points", xytext=(0,-15), ha='center', va='top', fontsize=30)

        # Define dynamic offsets for positioning real and imaginary plots for the current m_q
        # These values are relative to x_pos and are in data units of the main axes.
        effective_inset_plot_data_width_per_plot = 1.5 # width parameter for inset_axes, scaled by 3
        gap_between_real_imag = -2.5 # Gap between the real and imaginary plots, reduced from 0.05

        if m_q == 0:
            # For m=0, only real part, centered on x_pos
            current_real_plot_x_offset = 0
            current_imag_plot_x_offset = None # No imaginary plot
        else:
            # For m != 0, real and imaginary side-by-side, centered around x_pos
            # Total width of the pair: 2 * effective_inset_plot_data_width_per_plot + gap_between_real_imag
            # Half of this total width determines the shift from x_pos to center the pair.
            # The real plot will be at -(half_total_pair_width - half_plot_width)
            half_total_pair_width = (effective_inset_plot_data_width_per_plot + gap_between_real_imag / 2)
            current_real_plot_x_offset = -half_total_pair_width
            current_imag_plot_x_offset = half_total_pair_width

        psi_m = particle_in_ring_eigenfunction(phi, m_q, R)

        # --- Real Part Polar Plot ---
        real_part_for_coloring = np.real(psi_m)
        radial_coords_real = np.abs(real_part_for_coloring)

        # Create an inset polar subplot for the Real Part
        axins_real = inset_axes(ax, width=effective_inset_plot_data_width_per_plot, height=effective_inset_plot_data_width_per_plot,
                           loc='center',
                           bbox_to_anchor=(x_pos + current_real_plot_x_offset, energy + 0.6), # Data coordinates for anchor point, increased y-offset
                           bbox_transform=ax.transData,
                           borderpad=0,
                           axes_class=plt.PolarAxes)

        # Plotting with gradient color by plotting small segments
        for k in range(len(phi) - 1):
            axins_real.plot(phi[k:k+2], radial_coords_real[k:k+2],
                      color=cmap(global_norm(real_part_for_coloring[k])),
                      linewidth=2)

        axins_real.set_theta_zero_location('E') # Set 0 degrees to the East (right)
        axins_real.set_theta_direction(1)    # Clockwise direction
        axins_real.set_rlim(0, r_max_polar) # Radial axis starts from zero
        axins_real.set_xticks([]) # Hide polar ticks
        axins_real.set_yticks([]) # Hide polar ticks
        axins_real.patch.set_alpha(0) # Make background transparent
        # "Re" to the left of the plot (a title above it would collide with the level labels)
        axins_real.text(-0.08, 0.5, "Re", transform=axins_real.transAxes, ha='right', va='center', fontsize=14)

        # Add nodal lines for Real Part if m_q is not 0
        if m_q != 0:
            node_angles_real = []
            for n_val in range(2 * abs(m_q)):
                angle = (n_val + 0.5) * np.pi / abs(m_q)
                if 0 <= angle <= 2 * np.pi:
                    node_angles_real.append(angle)
            for node_angle in node_angles_real:
                axins_real.plot([node_angle, node_angle], [0, r_max_polar], color='purple', linestyle='-', linewidth=1)

        # --- Imaginary Part Polar Plot (only for m_q != 0) ---
        if m_q != 0 and current_imag_plot_x_offset is not None:
            imaginary_part_for_coloring = np.imag(psi_m)
            radial_coords_imag = np.abs(imaginary_part_for_coloring)

            # Create an inset polar subplot for the Imaginary Part
            axins_imag = inset_axes(ax, width=effective_inset_plot_data_width_per_plot, height=effective_inset_plot_data_width_per_plot,
                               loc='center',
                               bbox_to_anchor=(x_pos + current_imag_plot_x_offset, energy + 0.6), # Position to the right of real part, increased y-offset
                               bbox_transform=ax.transData,
                               borderpad=0,
                               axes_class=plt.PolarAxes)

            # Plotting with gradient color by plotting small segments
            for k in range(len(phi) - 1):
                axins_imag.plot(phi[k:k+2], radial_coords_imag[k:k+2],
                              color=cmap(global_norm(imaginary_part_for_coloring[k])),
                              linewidth=2)

            axins_imag.set_theta_zero_location('E') # Set 0 degrees to the East (right)
            axins_imag.set_theta_direction(1)    # Clockwise direction
            axins_imag.set_rlim(0, r_max_polar) # Radial axis starts from zero
            axins_imag.set_xticks([]) # Hide polar ticks
            axins_imag.set_yticks([]) # Hide polar ticks
            axins_imag.patch.set_alpha(0) # Make background transparent
            # "Im" to the right of the plot
            axins_imag.text(1.08, 0.5, "Im", transform=axins_imag.transAxes, ha='left', va='center', fontsize=14)

            # Add nodal lines for Imaginary Part
            node_angles_imag = []
            for n_val in range(2 * abs(m_q)):
                angle = n_val * np.pi / abs(m_q) # Nodes for sin(m*phi) are at n*pi/|m|
                if 0 <= angle <= 2 * np.pi:
                    node_angles_imag.append(angle)
            for node_angle in node_angles_imag:
                axins_imag.plot([node_angle, node_angle], [0, r_max_polar], color='purple', linestyle='-', linewidth=1)

    # Update max width needed for x-axis limits
    max_width_needed = max(max_width_needed, num_degeneracies * offset_step)

ax.set_ylabel('Energy') # Energy is now on the y-axis, increased font size
ax.set_xticks([]) # Hide x-axis ticks as it's just a scale
ax.set_yticks(unique_energies, y_tick_labels, fontsize=25) # Set custom y-axis ticks and labels
ax.grid(axis='y', linestyle='--', alpha=0.7) # Grid on y-axis
ax.set_ylim(min(unique_energies) - 0.5, max(unique_energies) + 1.5) # Adjust y-limits for better view
ax.set_xlim(-max_width_needed / 2, max_width_needed / 2) # Adjust x-limits dynamically
save_for_print(fig, 'PIR_levels', 4.05)
plt.show()